In [1]:
import pandas as pd
import numpy as np
import re

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.preprocessing import StandardScaler

#### Load Data

In [2]:
train_df = pd.read_parquet("../data/task_a/train.parquet")
val_df   = pd.read_parquet("../data/task_a/val.parquet")
eval_df  = pd.read_parquet("../data/task_a/test_sample.parquet")

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Eval:", len(eval_df))

Train: 500000
Val: 100000
Eval: 1000


#### Define Simple Structural Feature Extractor

In [3]:
def extract_structural_features(code):
    lines = code.split('\n')
    num_lines = len(lines)
    
    # Basic length features
    num_chars = len(code)
    avg_line_length = np.mean([len(l) for l in lines]) if num_lines > 0 else 0
    
    # Loop and conditional counts
    num_for = len(re.findall(r'\bfor\b', code))
    num_while = len(re.findall(r'\bwhile\b', code))
    num_if = len(re.findall(r'\bif\b', code))
    
    # Function indicators
    num_def = len(re.findall(r'\bdef\b', code))
    num_function = len(re.findall(r'\bfunction\b', code))
    
    # Braces and semicolons
    num_open_brace = code.count('{')
    num_close_brace = code.count('}')
    num_semicolon = code.count(';')
    
    # Comment indicators
    num_hash_comment = code.count('#')
    num_slash_comment = code.count('//')
    
    # Blank lines
    blank_lines = sum(1 for l in lines if l.strip() == "")
    
    # Indentation variation
    indent_levels = [len(l) - len(l.lstrip(' ')) for l in lines if l.strip() != ""]
    indent_std = np.std(indent_levels) if indent_levels else 0
    
    return [
        num_lines,
        num_chars,
        avg_line_length,
        num_for,
        num_while,
        num_if,
        num_def,
        num_function,
        num_open_brace,
        num_close_brace,
        num_semicolon,
        num_hash_comment,
        num_slash_comment,
        blank_lines,
        indent_std
    ]

#### Apply Feature Extraction

In [4]:
print("Extracting structural features for training...")
X_train = np.array(train_df['code'].apply(extract_structural_features).tolist())

print("Extracting structural features for validation...")
X_val = np.array(val_df['code'].apply(extract_structural_features).tolist())

print("Extracting structural features for evaluation...")
X_eval = np.array(eval_df['code'].apply(extract_structural_features).tolist())

y_train = train_df['label'].values
y_val   = val_df['label'].values
y_eval  = eval_df['label'].values

print("Feature shape:", X_train.shape)

Extracting structural features for training...
Extracting structural features for validation...
Extracting structural features for evaluation...
Feature shape: (500000, 15)


#### Scale Features

In [5]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_eval  = scaler.transform(X_eval)

#### Train Logistic Regression

In [6]:
model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

print("Training structural model...")
model.fit(X_train, y_train)

Training structural model...


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


#### Evaluate (IID)

In [7]:
val_preds = model.predict(X_val)

print("=== Validation Performance (Structural) ===")
print("Accuracy:", accuracy_score(y_val, val_preds))
print("F1:", f1_score(y_val, val_preds))
print(classification_report(y_val, val_preds))

=== Validation Performance (Structural) ===
Accuracy: 0.88551
F1: 0.8864625789625046
              precision    recall  f1-score   support

           0       0.85      0.92      0.88     47695
           1       0.92      0.85      0.89     52305

    accuracy                           0.89    100000
   macro avg       0.89      0.89      0.89    100000
weighted avg       0.89      0.89      0.89    100000



#### Evaluate (OOD)

In [8]:
eval_preds = model.predict(X_eval)

print("=== Evaluation Performance (Structural) ===")
print("Accuracy:", accuracy_score(y_eval, eval_preds))
print("F1:", f1_score(y_eval, eval_preds))
print(classification_report(y_eval, eval_preds))

=== Evaluation Performance (Structural) ===
Accuracy: 0.422
F1: 0.3837953091684435
              precision    recall  f1-score   support

           0       0.85      0.31      0.46       777
           1       0.25      0.81      0.38       223

    accuracy                           0.42      1000
   macro avg       0.55      0.56      0.42      1000
weighted avg       0.72      0.42      0.44      1000



All current models learn patterns specific to the training distribution (competitive programming style, limited languages, balanced labels). Evaluation data contains broader language diversity and domain variety, leading to systematic generalization failure.

## Diagnose Root Cause of OOD Collapse

### Hypothesis 1:

Is performance collapse due to cross-language shift?

If performance improves strongly,
problem = language shift.

If still poor,
problem = domain shift.

#### Filter Python Only

In [9]:
# Python-only training
train_py = train_df[train_df['language'] == 'Python']
val_py   = val_df[val_df['language'] == 'Python']
eval_py  = eval_df[eval_df['language'] == 'Python']

print(len(train_py), len(val_py), len(eval_py))

457306 91461 303


#### Use Character TF-IDF (Best lexical so far)

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer_py = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3,5),
    max_features=100000,
    min_df=5,
    sublinear_tf=True
)

X_train_py = vectorizer_py.fit_transform(train_py['code'])
X_val_py   = vectorizer_py.transform(val_py['code'])
X_eval_py  = vectorizer_py.transform(eval_py['code'])

y_train_py = train_py['label']
y_val_py   = val_py['label']
y_eval_py  = eval_py['label']

Case 1:
- Eval F1 improves strongly (e.g., >0.6)
- → Cross-language shift is dominant problem.

Case 2:
- Eval F1 still ~0.4
- → Domain shift is dominant problem.

#### Train + Evaluate

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

model_py = LogisticRegression(max_iter=1000, class_weight="balanced")
model_py.fit(X_train_py, y_train_py)

print("Validation F1 (Python only):",
      f1_score(y_val_py, model_py.predict(X_val_py)))

print("Evaluation F1 (Python only):",
      f1_score(y_eval_py, model_py.predict(X_eval_py)))

Validation F1 (Python only): 0.969106343441169
Evaluation F1 (Python only): 0.48598130841121495


Diagnostic Insight — Python-Only Evaluation

Restricting both training and evaluation to Python improved OOD F1 from ~0.38 to ~0.49.

This confirms that cross-language distribution shift is a major source of performance degradation.

However, performance remains substantially below in-domain levels (~0.97), indicating that domain and style shifts also contribute significantly to OOD collapse.

Thus, language mismatch is necessary but not sufficient to explain generalization failure.

Python AST Structural Experiment (Diagnostic Stage)

We previously observed that:

- Cross-language shift explains part of OOD degradation.
- Python-only evaluation improves F1 from ~0.38 to ~0.49.
- However, performance remains far below in-domain (~0.97).

This suggests domain/style shift persists even within Python.

We now test whether deeper structural features extracted from Python AST improve OOD robustness.

In [12]:
import ast
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.preprocessing import StandardScaler
from collections import Counter
import math

#### Define AST Feature Extractor

In [13]:
def extract_ast_features(code):
    try:
        tree = ast.parse(code)
    except:
        return [0]*8  # fallback if parsing fails
    
    node_types = []
    max_depth = 0
    
    def traverse(node, depth=0):
        nonlocal max_depth
        node_types.append(type(node).__name__)
        max_depth = max(max_depth, depth)
        
        for child in ast.iter_child_nodes(node):
            traverse(child, depth+1)
    
    traverse(tree)
    
    total_nodes = len(node_types)
    
    if total_nodes == 0:
        return [0]*8
    
    # Node type entropy
    counter = Counter(node_types)
    probs = [v/total_nodes for v in counter.values()]
    entropy = -sum(p * math.log(p + 1e-10) for p in probs)
    
    # Function count
    num_functions = sum(1 for n in node_types if n == "FunctionDef")
    
    # Loop count
    num_loops = sum(1 for n in node_types if n in ["For", "While"])
    
    # If statements
    num_ifs = sum(1 for n in node_types if n == "If")
    
    # Assignments
    num_assign = sum(1 for n in node_types if n == "Assign")
    
    # Return statements
    num_return = sum(1 for n in node_types if n == "Return")
    
    return [
        total_nodes,
        max_depth,
        entropy,
        num_functions,
        num_loops,
        num_ifs,
        num_assign,
        num_return
    ]

#### Filter Python Only

In [14]:
train_py = train_df[train_df['language'] == 'Python']
val_py   = val_df[val_df['language'] == 'Python']
eval_py  = eval_df[eval_df['language'] == 'Python']

#### Extract AST Features

In [15]:
print("Extracting AST features (train)...")
X_train_ast = np.array(train_py['code'].apply(extract_ast_features).tolist())

print("Extracting AST features (val)...")
X_val_ast = np.array(val_py['code'].apply(extract_ast_features).tolist())

print("Extracting AST features (eval)...")
X_eval_ast = np.array(eval_py['code'].apply(extract_ast_features).tolist())

y_train_ast = train_py['label'].values
y_val_ast   = val_py['label'].values
y_eval_ast  = eval_py['label'].values

Extracting AST features (train)...
Extracting AST features (val)...
Extracting AST features (eval)...


#### Scale Features

In [16]:
scaler = StandardScaler()
X_train_ast = scaler.fit_transform(X_train_ast)
X_val_ast   = scaler.transform(X_val_ast)
X_eval_ast  = scaler.transform(X_eval_ast)

#### Train Model

In [17]:
model_ast = LogisticRegression(max_iter=1000, class_weight="balanced")
model_ast.fit(X_train_ast, y_train_ast)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


####  Evaluate

In [18]:
print("Validation F1 (AST):",
      f1_score(y_val_ast, model_ast.predict(X_val_ast)))

print("Evaluation F1 (AST):",
      f1_score(y_eval_ast, model_ast.predict(X_eval_ast)))

Validation F1 (AST): 0.7207688338493292
Evaluation F1 (AST): 0.5314685314685315


AST Structural Diagnostic Result (Python Only)

AST-based structural features reduce in-domain performance (F1 ~0.72 vs ~0.97 for lexical), but improve OOD robustness (F1 ~0.53 vs ~0.49 for character TF-IDF).

This indicates that deep structural properties generalize better across domain shift than surface lexical patterns.

Thus, structural modeling provides improved robustness despite lower in-domain accuracy.

### Enhanced AST Structural Modeling

- Basic AST features improved OOD robustness (F1 ~0.53 vs ~0.49 lexical).
- We now extend feature extraction to capture richer structural properties including branching factor, node ratios, leaf statistics, and cyclomatic complexity proxies.
- The goal is to test whether deeper structural characterization further improves domain robustness.

In [19]:
def extract_enhanced_ast_features(code):
    try:
        tree = ast.parse(code)
    except:
        return [0]*15
    
    node_types = []
    depths = []
    child_counts = []
    
    def traverse(node, depth=0):
        node_types.append(type(node).__name__)
        depths.append(depth)
        
        children = list(ast.iter_child_nodes(node))
        child_counts.append(len(children))
        
        for child in children:
            traverse(child, depth+1)
    
    traverse(tree)
    
    total_nodes = len(node_types)
    if total_nodes == 0:
        return [0]*15
    
    counter = Counter(node_types)
    
    # Node type entropy
    probs = [v/total_nodes for v in counter.values()]
    entropy = -sum(p * math.log(p + 1e-10) for p in probs)
    
    # Depth statistics
    max_depth = max(depths)
    avg_depth = np.mean(depths)
    depth_std = np.std(depths)
    
    # Branching statistics
    avg_branching = np.mean(child_counts)
    max_branching = max(child_counts)
    
    # Node ratios
    num_functions = counter.get("FunctionDef", 0)
    num_loops = counter.get("For", 0) + counter.get("While", 0)
    num_ifs = counter.get("If", 0)
    num_assign = counter.get("Assign", 0)
    num_return = counter.get("Return", 0)
    
    # Leaf nodes
    num_leaves = sum(1 for c in child_counts if c == 0)
    leaf_ratio = num_leaves / total_nodes
    
    # Cyclomatic complexity proxy
    cyclomatic = num_loops + num_ifs + 1
    
    return [
        total_nodes,
        entropy,
        max_depth,
        avg_depth,
        depth_std,
        avg_branching,
        max_branching,
        num_functions,
        num_loops,
        num_ifs,
        num_assign,
        num_return,
        num_leaves,
        leaf_ratio,
        cyclomatic
    ]

#### Extract Features (Python Only)

In [20]:
print("Extracting enhanced AST features (train)...")
X_train_enh = np.array(train_py['code'].apply(extract_enhanced_ast_features).tolist())

print("Extracting enhanced AST features (val)...")
X_val_enh = np.array(val_py['code'].apply(extract_enhanced_ast_features).tolist())

print("Extracting enhanced AST features (eval)...")
X_eval_enh = np.array(eval_py['code'].apply(extract_enhanced_ast_features).tolist())

Extracting enhanced AST features (train)...
Extracting enhanced AST features (val)...
Extracting enhanced AST features (eval)...


#### Scale

In [21]:
scaler_enh = StandardScaler()
X_train_enh = scaler_enh.fit_transform(X_train_enh)
X_val_enh   = scaler_enh.transform(X_val_enh)
X_eval_enh  = scaler_enh.transform(X_eval_enh)

#### Train Model

In [22]:
model_enh = LogisticRegression(max_iter=1000, class_weight="balanced")
model_enh.fit(X_train_enh, y_train_ast)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


#### Evaluate

In [23]:
print("Validation F1 (Enhanced AST):",
      f1_score(y_val_ast, model_enh.predict(X_val_enh)))

print("Evaluation F1 (Enhanced AST):",
      f1_score(y_eval_ast, model_enh.predict(X_eval_enh)))

Validation F1 (Enhanced AST): 0.7226497068676717
Evaluation F1 (Enhanced AST): 0.5846153846153846


#### Enhanced AST Structural Insight

- Enhancing AST features improves Python-only OOD F1 from ~0.53 to ~0.585.
- This demonstrates that deeper structural characterization increases robustness under domain shift.
- Although in-domain performance remains lower than lexical models, structural representations generalize better to unseen evaluation distributions.
- This supports the hypothesis that machine-generation signals are better captured through structural invariants rather than lexical patterns.